In [1]:
%load_ext lab_black

import pandas as pd
import numpy as np
import polars as pl
from tqdm import tqdm
import os
import gc
import warnings

warnings.filterwarnings("ignore")

pd.set_option("max_rows", 500)

In [2]:
catalog.exists("sample_submission")

True

In [3]:
catalog.exists("train_as_df"), catalog.exists("test_as_df")

(True, True)

In [4]:
%%time

df_train = catalog.load("train_as_df")
df_test = catalog.load("test_as_df")

df_train = pl.from_pandas(df_train)
df_test = pl.from_pandas(df_test)

[12/05/22 08:06:54] INFO     Loading data from 'train_as_df' (ParquetDataSet)...                ]8;id=566956;file:///media/mikhail/disk/ML/compet/kaggle/otto-recommender-system/venv_otto/lib/python3.7/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=743689;file:///media/mikhail/disk/ML/compet/kaggle/otto-recommender-system/venv_otto/lib/python3.7/site-packages/kedro/io/data_catalog.py#344\344]8;;\

[12/05/22 08:07:11] INFO     Loading data from 'test_as_df' (ParquetDataSet)...                 ]8;id=93395;file:///media/mikhail/disk/ML/compet/kaggle/otto-recommender-system/venv_otto/lib/python3.7/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=601532;file:///media/mikhail/disk/ML/compet/kaggle/otto-recommender-system/venv_otto/lib/python3.7/site-packages/kedro/io/data_catalog.py#344\344]8;;\

CPU times: user 27.5 s, sys: 16 s, total: 43.5 s
Wall time: 25.5 s


In [5]:
def build_n_sessions(df, trs_ms=1000 * 60 * 60 * 2):
    df = df.sort(["session", "ts"])
    df_diffs = df.groupby("session", maintain_order=True).agg(
        pl.col("ts").diff().alias("d")
    )
    df_diffs = df_diffs.explode("d")
    df = df.with_column(
        pl.Series(name="d", values=list(df_diffs.select("d"))[0]).fill_null(0)
    )

    del df_diffs
    gc.collect()

    df = df.select(
        [
            pl.col("*"),
            pl.col("d").apply(lambda x: (x > trs_ms) * 1).alias("session_changed"),
        ]
    )

    df_n_session = df.groupby("session", maintain_order=True).agg(
        pl.col("session_changed").cumsum().alias("n_session")
    )
    df_n_session = df_n_session.explode("n_session")
    df = df.with_column(
        pl.Series(name="n_session", values=list(df_n_session.select("n_session"))[0])
    )

    del df_n_session
    gc.collect()

    for col in ["d", "session_changed"]:
        df.drop_in_place(col)

    df = df.rename({"session": "user"})

    return df

In [6]:
%%time

df_train = build_n_sessions(df_train)
df_test = build_n_sessions(df_test)

CPU times: user 6min 13s, sys: 1min 4s, total: 7min 18s
Wall time: 1min 22s


In [7]:
df_train

user,aid,ts,type,n_session
i64,i64,i64,str,i64
0,1517085,1659304800025,"""clicks""",0
0,1563459,1659304904511,"""clicks""",0
0,1309446,1659367439426,"""clicks""",1
0,16246,1659367719997,"""clicks""",1
0,1781822,1659367871344,"""clicks""",1
0,1152674,1659367885796,"""clicks""",1
0,1649869,1659369893840,"""carts""",1
0,461689,1659369898050,"""carts""",1
0,305831,1659370027105,"""orders""",1


In [8]:
df_test

user,aid,ts,type,n_session
i64,i64,i64,str,i64
12899779,59625,1661724000278,"""clicks""",0
12899780,1142000,1661724000378,"""clicks""",0
12899780,582732,1661724058352,"""clicks""",0
12899780,973453,1661724109199,"""clicks""",0
12899780,736515,1661724136868,"""clicks""",0
12899780,1142000,1661724155248,"""clicks""",0
12899781,141736,1661724000559,"""clicks""",0
12899781,199008,1661724022851,"""clicks""",0
12899781,57315,1661724170835,"""clicks""",0


In [9]:
%%time

df_train.write_parquet('../data/02_intermediate/df_train_n_session.parquet')

CPU times: user 26.2 s, sys: 10.2 s, total: 36.4 s
Wall time: 30.4 s


In [10]:
%%time

df_test.write_parquet('../data/02_intermediate/df_test_n_session.parquet')

CPU times: user 885 ms, sys: 79.8 ms, total: 965 ms
Wall time: 926 ms
